# Data Pre-processing of NSW Electricity Demand

This notebook prepares the electricity demand dataset for modelling. The main steps are:
1. Standardise datetime formats across datasets
2. Transform forecast data into multi-horizon format (P01–P48)
3. Merge demand, temperature, and forecast datasets
4. Handle missing values and restore temporal continuity
5. Construct lag-based features to capture temporal patterns
6. Split the dataset into training, validation, and test sets
7. Apply feature scaling for machine learning and deep learning models



## Section 1 - Objective

The purpose of this notebook is to prepare the NSW electricity demand dataset for forecasting models.

This includes:
1. converting the datetime field into a proper time index,
2. checking missing values,
3. confirming alignment across datasets,
4. selecting modelling variables,
5. splitting the data into training and testing sets.

## Section 2 - Import Libraries and Load Raw Data

In [1]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

Cloning into 'capstone_project_GroupA'...
remote: Enumerating objects: 1632, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 1632 (delta 36), reused 38 (delta 13), pack-reused 1540 (from 3)
Receiving objects: 100% (1632/1632), 414.49 MiB | 14.26 MiB/s, done.
Resolving deltas: 100% (897/897), done.
Updating files: 100% (115/115), done.


In [47]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

repo_path = "capstone_project_GroupA"
nsw_path = os.path.join(repo_path, "data", "NSW")

part_a = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partaa")
part_b = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partab")
forecast_zip = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip")

with open(forecast_zip, "wb") as outfile:
    for p in [part_a, part_b]:
        with open(p, "rb") as infile:
            outfile.write(infile.read())

df_demand = pd.read_csv(os.path.join(nsw_path, "totaldemand_nsw.csv.zip"))
df_temp = pd.read_csv(os.path.join(nsw_path, "temperature_nsw.csv.zip"))
df_forecast = pd.read_csv(os.path.join(nsw_path, "forecastdemand_nsw.csv.zip"))

print("Demand shape:", df_demand.shape)
print("Temp shape:", df_temp.shape)
print("Forecast shape:", df_forecast.shape)

Demand shape: (196513, 3)
Temp shape: (220326, 3)
Forecast shape: (10906019, 6)


## Section 3. Convert Datetime Format
Purpose of Section 3 is to ensure all datasets share a consistent 30-minute timestamp format.  We converted "LASTCHANGED" in table df_forecast to datetime and round it so it can align with the demand timestamps.

In [48]:
df_demand["DATETIME"] = pd.to_datetime(df_demand["DATETIME"], dayfirst=True)
df_temp["DATETIME"] = pd.to_datetime(df_temp["DATETIME"], dayfirst=True)
df_forecast["LASTCHANGED"] = pd.to_datetime(df_forecast["LASTCHANGED"])
df_forecast["TARGET_DATETIME"] = (
    df_forecast["LASTCHANGED"] + pd.to_timedelta(df_forecast["PERIODID"] * 30, unit="m")
)
print(df_demand.dtypes)
print(df_temp.dtypes)
print(df_forecast.dtypes)

DATETIME       datetime64[ns]
TOTALDEMAND           float64
REGIONID               object
dtype: object
LOCATION               object
DATETIME       datetime64[ns]
TEMPERATURE           float64
dtype: object
PREDISPATCHSEQNO             int64
REGIONID                    object
PERIODID                     int64
FORECASTDEMAND             float64
LASTCHANGED         datetime64[ns]
DATETIME                    object
TARGET_DATETIME     datetime64[ns]
dtype: object


In [49]:
print(df_forecast.head())

   PREDISPATCHSEQNO REGIONID  PERIODID  FORECASTDEMAND         LASTCHANGED  \
0        2009123018     NSW1        71         7832.04 2009-12-30 12:31:49   
1        2009123019     NSW1        70         7832.04 2009-12-30 13:01:43   
2        2009123020     NSW1        69         7832.03 2009-12-30 13:31:36   
3        2009123021     NSW1        68         7832.03 2009-12-30 14:01:44   
4        2009123022     NSW1        67         7830.96 2009-12-30 14:31:35   

              DATETIME     TARGET_DATETIME  
0  2010-01-01 00:00:00 2010-01-01 00:01:49  
1  2010-01-01 00:00:00 2010-01-01 00:01:43  
2  2010-01-01 00:00:00 2010-01-01 00:01:36  
3  2010-01-01 00:00:00 2010-01-01 00:01:44  
4  2010-01-01 00:00:00 2010-01-01 00:01:35  


## Section 4. Aggregate Forecast Data by 30-Minute Interval

Purpose of Section 4 is to convert forecast data into a structured multi-horizon format (P01–P48), to enable horizon-wise analysis.
There may be multiple forecast rows within the same rounded timestamp, so aggregate them using mean.

PERIODID indicates the forecast lead time in 30-minute steps. We retain PERIODID 1–48 so that each target datetime contains forecast values from 0.5 hours ahead up to 24 hours ahead.

After pivoting:
- P01 represents the shortest forecast lead time (from 1/2 hr ago to that datetime)
- P48 represents the longest forecast lead time within the 24-hour window (from 12 hrs ago to that datetime)

This wide format makes it easier to:
1. compare forecast accuracy across horizons,
2. analyse how the provided forecast deteriorates as horizon increases,
3. align forecast horizons with the modelling and evaluation framework.

In [50]:
# Rounding down datetime
df_demand["DATETIME"] = pd.to_datetime(df_demand["DATETIME"]).dt.floor("30min")
df_temp["DATETIME"] = pd.to_datetime(df_temp["DATETIME"]).dt.floor("30min")
df_temp_30min = (
    df_temp
    .groupby("DATETIME", as_index=False)["TEMPERATURE"]
    .mean()
)
df_forecast["TARGET_DATETIME"] = pd.to_datetime(df_forecast["TARGET_DATETIME"]).dt.floor("30min")

# Keep only PERIODID 1–48
df_forecast_48 = df_forecast[df_forecast["PERIODID"].between(1, 48)].copy()

# Keep only relevant columns
df_forecast_48 = df_forecast_48[["TARGET_DATETIME", "PERIODID", "FORECASTDEMAND"]]

# Rename TARGET_DATETIME → DATETIME
df_forecast_48 = df_forecast_48.rename(columns={"TARGET_DATETIME": "DATETIME"})

# Handle duplicates (same DATETIME + PERIODID)
df_forecast_48 = (
    df_forecast_48
    .groupby(["DATETIME", "PERIODID"], as_index=False)["FORECASTDEMAND"]
    .mean()
)

# Pivot to wide format → P01, P02, ... P48
df_forecast_wide = df_forecast_48.pivot(
    index="DATETIME",
    columns="PERIODID",
    values="FORECASTDEMAND"
)

# Rename columns → P01, P02, ..., P48
df_forecast_wide.columns = [f"P{int(c):02d}" for c in df_forecast_wide.columns]

# Build full datetime index
forecast_index = pd.date_range(
    start=df_demand["DATETIME"].min(),
    end=df_demand["DATETIME"].max(),
    freq="30min"
)

df_forecast_wide = (
    df_forecast_wide
    .reindex(forecast_index)
)

df_forecast_wide.index.name = "DATETIME"

# Interpolate missing values
# Interpolate each horizon separately
df_forecast_wide = df_forecast_wide.interpolate(method="time")

# Fill any edge missing values
df_forecast_wide = df_forecast_wide.ffill().bfill()

# Reset index
df_forecast_wide = df_forecast_wide.reset_index()

# Check result
print(df_forecast_wide.head())

             DATETIME      P01      P02      P03      P04      P05      P06  \
0 2010-01-01 00:00:00  7999.11  7789.50  7812.04  7835.91  7829.69  7813.10   
1 2010-01-01 00:30:00  7596.21  7603.17  7595.34  7619.95  7636.28  7637.79   
2 2010-01-01 01:00:00  7380.70  7304.27  7307.44  7298.55  7326.25  7337.51   
3 2010-01-01 01:30:00  7022.05  7046.39  6976.87  6980.06  6970.82  7000.78   
4 2010-01-01 02:00:00  6682.92  6680.15  6714.37  6655.83  6657.09  6646.27   

       P07      P08      P09  ...      P39      P40      P41      P42  \
0  7788.74  7824.68  7824.39  ...  7820.53  7819.75  7818.89  7818.86   
1  7623.95  7628.41  7621.97  ...  7713.04  7713.19  7713.25  7711.33   
2  7340.77  7333.37  7333.97  ...  7488.44  7488.53  7488.71  7488.61   
3  7004.96  7002.24  7006.86  ...  7136.84  7137.88  7136.97  7137.85   
4  6676.93  6672.93  6676.41  ...  6802.50  6801.26  6800.29  6800.40   

       P43      P44      P45      P46      P47      P48  
0  7816.68  7816.32  7824.40

In addition to df_forecast_wide (P01–P48), we extract the PERIODID = 48 forecast as a standalone series. This is used for visual comparison and diagnostic plots of the provided 1-day-ahead forecast against actual demand. It is not a duplicate of the main preprocessing pipeline, but a simplified series created specifically for plotting.

In [51]:
# Build a separate P48-only forecast series for plotting and diagnostic analysis.
# This is used to visualise the provided 1-day-ahead forecast separately.
df_forecast_p48 = df_forecast[df_forecast["PERIODID"] == 48].copy()

# Keep only datetime + forecast value
df_forecast_p48 = (
    df_forecast_p48[["TARGET_DATETIME", "FORECASTDEMAND"]]
    .rename(columns={"TARGET_DATETIME": "DATETIME"})
    .sort_values("DATETIME")
)

# If multiple rows map to the same half-hour timestamp, average them
df_forecast_p48 = (
    df_forecast_p48
    .groupby("DATETIME", as_index=False)["FORECASTDEMAND"]
    .mean()
)

# Build a complete 30-minute forecast timeline covering demand range
forecast_index = pd.date_range(
    start=df_demand["DATETIME"].min(),
    end=df_demand["DATETIME"].max(),
    freq="30min"
)

# Reindex and interpolate missing PERIODID=48 forecasts
df_forecast_p48 = (
    df_forecast_p48
    .set_index("DATETIME")
    .reindex(forecast_index)
)

df_forecast_p48.index.name = "DATETIME"

# Time-based interpolation for missing forecast values
df_forecast_p48["FORECASTDEMAND"] = (
    df_forecast_p48["FORECASTDEMAND"]
    .interpolate(method="time")
    .ffill()
    .bfill()
)

df_forecast_p48 = df_forecast_p48.reset_index()

print(df_forecast_p48.head())
print(df_temp_30min.head())

             DATETIME  FORECASTDEMAND
0 2010-01-01 00:00:00         7822.38
1 2010-01-01 00:30:00         7715.68
2 2010-01-01 01:00:00         7482.56
3 2010-01-01 01:30:00         7129.32
4 2010-01-01 02:00:00         6800.73
             DATETIME  TEMPERATURE
0 2010-01-01 00:00:00         23.1
1 2010-01-01 00:30:00         22.8
2 2010-01-01 01:00:00         22.6
3 2010-01-01 01:30:00         22.5
4 2010-01-01 02:00:00         22.5


## Section 5. Merge the Three Datasets

Purpose of Section 5 is to combine demand, temperature, and forecast data into a single dataset. We are merging data using DATETIME as the key.

In [52]:
df_merged = (
    df_demand
    .merge(df_temp_30min, on="DATETIME", how="inner")
    .merge(df_forecast_wide, on="DATETIME", how="left")
)
df_merged.head()

,DATETIME,TOTALDEMAND,REGIONID,TEMPERATURE,P01,P02,P03,P04,P05,P06,...,P39,P40,P41,P42,P43,P44,P45,P46,P47,P48
0,2010-01-01 00:00:00,8038.00,NSW1,23.1,7999.11,7789.50,7812.04,7835.91,7829.69,7813.10,...,7820.53,7819.75,7818.89,7818.86,7816.68,7816.32,7824.40,7824.16,7824.56,7822.38
1,2010-01-01 00:30:00,7809.31,NSW1,22.8,7596.21,7603.17,7595.34,7619.95,7636.28,7637.79,...,7713.04,7713.19,7713.25,7711.33,7712.30,7709.95,7709.87,7716.82,7716.52,7715.68
2,2010-01-01 01:00:00,7483.69,NSW1,22.6,7380.70,7304.27,7307.44,7298.55,7326.25,7337.51,...,7488.44,7488.53,7488.71,7488.61,7489.25,7489.54,7488.59,7489.58,7484.49,7482.56
3,2010-01-01 01:30:00,7117.23,NSW1,22.5,7022.05,7046.39,6976.87,6980.06,6970.82,7000.78,...,7136.84,7137.88,7136.97,7137.85,7137.80,7139.10,7139.01,7137.02,7137.98,7129.32
4,2010-01-01 02:00:00,6812.03,NSW1,22.5,6682.92,6680.15,6714.37,6655.83,6657.09,6646.27,...,6802.50,6801.26,6800.29,6800.40,6801.60,6802.84,6804.55,6804.58,6802.68,6800.73


## Section 6. Sort and Set Time Index

Purpose of Section 6 is to sort data and set time index.  Sorting and setting time index is necessary because time-series models require an ordered time index.

In [53]:
df_merged = df_merged.sort_values("DATETIME") # Sort by time
df_merged = df_merged.set_index("DATETIME")   # Set as time index
df_merged = df_merged.asfreq("30min")         # Ensure frequency is set
df_merged.head()

,TOTALDEMAND,REGIONID,TEMPERATURE,P01,P02,P03,P04,P05,P06,P07,...,P39,P40,P41,P42,P43,P44,P45,P46,P47,P48
DATETIME,,,,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,8038.00,NSW1,23.1,7999.11,7789.50,7812.04,7835.91,7829.69,7813.10,7788.74,...,7820.53,7819.75,7818.89,7818.86,7816.68,7816.32,7824.40,7824.16,7824.56,7822.38
2010-01-01 00:30:00,7809.31,NSW1,22.8,7596.21,7603.17,7595.34,7619.95,7636.28,7637.79,7623.95,...,7713.04,7713.19,7713.25,7711.33,7712.30,7709.95,7709.87,7716.82,7716.52,7715.68
2010-01-01 01:00:00,7483.69,NSW1,22.6,7380.70,7304.27,7307.44,7298.55,7326.25,7337.51,7340.77,...,7488.44,7488.53,7488.71,7488.61,7489.25,7489.54,7488.59,7489.58,7484.49,7482.56
2010-01-01 01:30:00,7117.23,NSW1,22.5,7022.05,7046.39,6976.87,6980.06,6970.82,7000.78,7004.96,...,7136.84,7137.88,7136.97,7137.85,7137.80,7139.10,7139.01,7137.02,7137.98,7129.32
2010-01-01 02:00:00,6812.03,NSW1,22.5,6682.92,6680.15,6714.37,6655.83,6657.09,6646.27,6676.93,...,6802.50,6801.26,6800.29,6800.40,6801.60,6802.84,6804.55,6804.58,6802.68,6800.73


## Section 7. Check Missing Values

Purpose of Section 7 is to identify and resolve missing values introduced during merging and reindexing.

In [54]:
df_merged.isnull().sum()

,0
TOTALDEMAND,559
REGIONID,559
TEMPERATURE,559
P01,559
P02,559
P03,559
P04,559
P05,559
P06,559
P07,559


The merged dataset was checked for missing values using df_merged.isnull().sum().
Missing values are present after merging and setting a complete 30-minute frequency. These gaps arise because some timestamps are missing or not perfectly aligned across the demand, temperature, and forecast datasets. In the next step, these missing values are handled using time-based interpolation together with forward and backward filling to restore temporal continuity.

## Section 8. Remove Unnecessary Columns



The REGIONID variable contained only a single value (NSW1) across the entire dataset and therefore did not provide any predictive information. Since it has no variation, it does not provide any predictive information for modelling and is therefore removed.  It is because including constant features may introduce unnecessary noise and increase computational overhead without improving model performance.

In [56]:
df_merged["REGIONID"].unique()

array(['NSW1', nan], dtype=object)

In [34]:
df_merged = df_merged.drop(columns=["REGIONID"])

## Section 9. Reindex to Complete 30-Minute Timeline

The time index was examined to ensure continuity at the 30-minute frequency.
A total of 592 timestamps were missing from the expected sequence.
The dataset was therefore reindexed to a complete 30-minute timeline to ensure correct alignment when generating lag features.

In [35]:
# Section: Check missing time points

# expected full 30-min timeline
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# find missing timestamps
missing_timepoints = full_index.difference(df_merged.index)

print("Expected number of time points:", len(full_index))
print("Actual number of time points:", len(df_merged.index))
print("Number of missing time points:", len(missing_timepoints))

# preview first few missing timestamps
print(missing_timepoints[:20])

Expected number of time points: 196513
Actual number of time points: 196513
Number of missing time points: 0
DatetimeIndex([], dtype='datetime64[ns]', freq='30min')


In [36]:
time_gaps = df_merged.index.to_series().diff().value_counts().sort_index()
print(time_gaps)

DATETIME
0 days 00:30:00    196512
Name: count, dtype: int64


In [37]:
# Create full 30-min time index
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# Reindex to full timeline
df_merged = df_merged.reindex(full_index)
df_merged.index.name = "DATETIME"

# =========================================================
# Handle missing values
# =========================================================

# Interpolate numeric columns using time method
numeric_cols = ["TOTALDEMAND", "TEMPERATURE",
 "P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10",
 "P11", "P12", "P13", "P14", "P15", "P16", "P17", "P18", "P19", "P20",
 "P21", "P22", "P23", "P24", "P25", "P26", "P27", "P28", "P29", "P30",
 "P31", "P32", "P33", "P34", "P35", "P36", "P37", "P38", "P39", "P40",
 "P41", "P42", "P43", "P44", "P45", "P46", "P47", "P48"]

df_merged[numeric_cols] = (
    df_merged[numeric_cols]
    .interpolate(method="time")
    .ffill()
    .bfill()
)

# Final check
print(df_merged.isnull().sum())

TOTALDEMAND    0
TEMPERATURE    0
P01            0
P02            0
P03            0
P04            0
P05            0
P06            0
P07            0
P08            0
P09            0
P10            0
P11            0
P12            0
P13            0
P14            0
P15            0
P16            0
P17            0
P18            0
P19            0
P20            0
P21            0
P22            0
P23            0
P24            0
P25            0
P26            0
P27            0
P28            0
P29            0
P30            0
P31            0
P32            0
P33            0
P34            0
P35            0
P36            0
P37            0
P38            0
P39            0
P40            0
P41            0
P42            0
P43            0
P44            0
P45            0
P46            0
P47            0
P48            0
dtype: int64


After interpolation and forward/backward filling, no missing values remain in the numeric variables used for analysis and modelling. Missing numeric values in TOTALDEMAND, TEMPERATURE, and Forecast with various horizons P01, P02, ..., P48 were filled using time-based interpolation. This method estimates missing observations using surrounding values while preserving the temporal structure of the data.

Missing values are handled using time-based interpolation to preserve temporal continuity, which is appropriate for high-frequency time series such as electricity demand. Forward and backward filling are applied to address edge cases where interpolation is not sufficient.

While interpolation helps maintain continuity, it may introduce smoothing effects. However, this is acceptable given the relatively small proportion of missing values and the strong temporal structure of the data.

In [38]:
numeric_cols = ["TOTALDEMAND", "TEMPERATURE",
 "P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10",
 "P11", "P12", "P13", "P14", "P15", "P16", "P17", "P18", "P19", "P20",
 "P21", "P22", "P23", "P24", "P25", "P26", "P27", "P28", "P29", "P30",
 "P31", "P32", "P33", "P34", "P35", "P36", "P37", "P38", "P39", "P40",
 "P41", "P42", "P43", "P44", "P45", "P46", "P47", "P48"]

df_merged[numeric_cols] = df_merged[numeric_cols].interpolate(method="time")

print(df_merged[numeric_cols].isnull().sum())

TOTALDEMAND    0
TEMPERATURE    0
P01            0
P02            0
P03            0
P04            0
P05            0
P06            0
P07            0
P08            0
P09            0
P10            0
P11            0
P12            0
P13            0
P14            0
P15            0
P16            0
P17            0
P18            0
P19            0
P20            0
P21            0
P22            0
P23            0
P24            0
P25            0
P26            0
P27            0
P28            0
P29            0
P30            0
P31            0
P32            0
P33            0
P34            0
P35            0
P36            0
P37            0
P38            0
P39            0
P40            0
P41            0
P42            0
P43            0
P44            0
P45            0
P46            0
P47            0
P48            0
dtype: int64


## Section 10. Compute RMSE and Total Squared Error per row

Purpose of Section 10 is to evaluate model performance using room mean square error (RMSE) and total squared error, and to capture temporal dependencies using lagged demand values.

In [39]:
import numpy as np

# Define forecast columns
p_cols = [f"P{i:02d}" for i in range(1, 49)]

# Compute squared errors
sq_errors = (df_merged[p_cols].sub(df_merged["TOTALDEMAND"], axis=0)) ** 2

# (1) RMSE per datetime
df_merged["RMSE_48"] = np.sqrt(sq_errors.mean(axis=1))

# (2) Total squared error (sum of squared errors)
df_merged["TSE_48"] = sq_errors.sum(axis=1)

# Optional check
df_merged[["TOTALDEMAND", "RMSE_48", "TSE_48"]].head()

,TOTALDEMAND,RMSE_48,TSE_48
DATETIME,,,
2010-01-01 00:00:00,8038.00,217.708105,2.275047e+06
2010-01-01 00:30:00,7809.31,155.946782,1.167331e+06
2010-01-01 01:00:00,7483.69,114.015064,6.239729e+05
2010-01-01 01:30:00,7117.23,87.146686,3.645382e+05
2010-01-01 02:00:00,6812.03,105.817678,5.374743e+05


## Section 11. Add Historical Demand Fields

Because the data is every 1/2 hour:

1 day ago = 24 hrs x 2 = 48 rows before

1 week ago = 48 x 7 = 336 rows before

1 year ago = 48 x 365 = 17,520 rows before

In [45]:
df_merged["demand_1_day_ago"] = df_merged["TOTALDEMAND"].shift(48)
df_merged["demand_1_week_ago"] = df_merged["TOTALDEMAND"].shift(336)
df_merged["demand_1_year_ago"] = df_merged["TOTALDEMAND"].shift(17520)
df_merged.head(1000000)

,TOTALDEMAND,TEMPERATURE,P01,P02,P03,P04,P05,P06,P07,P08,...,P44,P45,P46,P47,P48,RMSE_48,TSE_48,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,8038.00,23.10,7999.11,7789.50,7812.04,7835.91,7829.69,7813.10,7788.74,7824.68,...,7816.32,7824.40,7824.16,7824.56,7822.38,217.708105,2.275047e+06,NaN,NaN,NaN
2010-01-01 00:30:00,7809.31,22.80,7596.21,7603.17,7595.34,7619.95,7636.28,7637.79,7623.95,7628.41,...,7709.95,7709.87,7716.82,7716.52,7715.68,155.946782,1.167331e+06,NaN,NaN,NaN
2010-01-01 01:00:00,7483.69,22.60,7380.70,7304.27,7307.44,7298.55,7326.25,7337.51,7340.77,7333.37,...,7489.54,7488.59,7489.58,7484.49,7482.56,114.015064,6.239729e+05,NaN,NaN,NaN
2010-01-01 01:30:00,7117.23,22.50,7022.05,7046.39,6976.87,6980.06,6970.82,7000.78,7004.96,7002.24,...,7139.10,7139.01,7137.02,7137.98,7129.32,87.146686,3.645382e+05,NaN,NaN,NaN
2010-01-01 02:00:00,6812.03,22.50,6682.92,6680.15,6714.37,6655.83,6657.09,6646.27,6676.93,6672.93,...,6802.84,6804.55,6804.58,6802.68,6800.73,105.817678,5.374743e+05,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-03-17 22:00:00,7419.77,19.70,7409.33,7412.81,7423.73,7442.55,7388.57,7418.10,7422.29,7397.60,...,7290.65,7290.58,7290.56,7288.56,7284.49,97.171391,4.532294e+05,7373.83,7642.24,7480.90
2021-03-17 22:30:00,7417.91,19.50,7422.63,7379.27,7351.76,7367.35,7379.97,7335.37,7356.95,7354.44,...,7251.02,7242.60,7242.46,7240.27,7240.18,145.427769,1.015163e+06,7345.78,7567.36,7465.50
2021-03-17 23:00:00,7287.32,19.05,7313.13,7316.62,7283.91,7248.31,7262.09,7273.77,7230.04,7243.40,...,7148.80,7147.34,7146.38,7146.41,7145.45,111.101969,5.924951e+05,7218.99,7461.71,7387.71


Rows with NaN entries in demand_1_day_ago, demand_1_week_ago, and demand_1_year_ago are removed to ensure that all lagged demand features are available for model training. These missing values occur at the beginning of the dataset where sufficient historical observations do not exist to compute the lag features.

In [41]:
df_model = df_merged.dropna(subset=[
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
])
df_model.head(1000000)

,TOTALDEMAND,TEMPERATURE,P01,P02,P03,P04,P05,P06,P07,P08,...,P44,P45,P46,P47,P48,RMSE_48,TSE_48,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,,,,,,,,,,,,,,,,,,
2010-07-02 12:00:00,11409.84,11.00,11362.90,11359.51,10955.30,10939.99,10802.79,10699.44,10707.36,10700.62,...,10762.04,10762.32,10762.16,10778.89,10385.83,637.616980,1.951466e+07,10035.63,9766.12,8038.00
2010-07-02 12:30:00,11276.77,11.50,11174.27,11173.78,11160.88,10790.56,10776.94,10650.20,10575.17,10569.95,...,10536.77,10536.06,10536.13,10535.94,10537.08,706.576978,2.396405e+07,9686.17,9521.15,7809.31
2010-07-02 13:00:00,11208.75,11.60,11039.89,11025.54,11027.53,11014.57,10666.67,10655.04,10530.05,10472.89,...,10357.23,10360.35,10359.41,10359.74,10359.19,784.498292,2.954100e+07,9513.42,9318.45,7483.69
2010-07-02 13:30:00,11184.47,11.80,10943.71,10937.73,10929.33,10936.32,10927.45,10540.39,10534.17,10414.15,...,10241.98,10248.59,10249.17,10248.50,10247.97,862.789475,3.573147e+07,9255.59,9108.57,7117.23
2010-07-02 14:00:00,11141.20,12.00,11163.10,10858.43,10855.01,10843.98,10848.74,10836.88,10431.05,10421.55,...,10114.91,10117.20,10119.65,10120.57,10119.70,927.314578,4.127579e+07,9019.79,8863.47,6812.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-03-17 22:00:00,7419.77,19.70,7409.33,7412.81,7423.73,7442.55,7388.57,7418.10,7422.29,7397.60,...,7290.65,7290.58,7290.56,7288.56,7284.49,97.171391,4.532294e+05,7695.32,7466.22,6591.28
2021-03-17 22:30:00,7417.91,19.50,7422.63,7379.27,7351.76,7367.35,7379.97,7335.37,7356.95,7354.44,...,7251.02,7242.60,7242.46,7240.27,7240.18,145.427769,1.015163e+06,7595.23,7374.06,6452.51
2021-03-17 23:00:00,7287.32,19.05,7313.13,7316.62,7283.91,7248.31,7262.09,7273.77,7230.04,7243.40,...,7148.80,7147.34,7146.38,7146.41,7145.45,111.101969,5.924951e+05,7537.24,7344.92,6372.17


## Section 12. Train / Validation / Test Split

Purpose of Section 12 is to create training, validation, and test sets for model development.

In [42]:
n = len(df_model)

base_features = ["TOTALDEMAND",
            "demand_1_day_ago",
            "demand_1_week_ago",
            "demand_1_year_ago",
            "TEMPERATURE",
            "RMSE_48",
            "TSE_48"]

train_size = int(n * 0.7)
val_size = int(n * 0.1)

train       = df_model[base_features].iloc[:train_size]
validation  = df_model[base_features].iloc[train_size:train_size + val_size]
test        = df_model.iloc[train_size + val_size:]

The dataset is split chronologically into 70% training, 10% validation, and 20% test sets. This split is consistent with prior studies in time series forecasting, where a larger training set is preferred to improve model learning, while retaining a validation set for hyperparameter tuning and a separate test set for final evaluation.

Compared to a 60/20/20 split, this configuration allocates more data for model training, which is beneficial for complex models such as LSTM and transformer-based architectures.

## Section 13. Feature Scaling

In [43]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

base_features = ["TOTALDEMAND",
            "demand_1_day_ago",
            "demand_1_week_ago",
            "demand_1_year_ago",
            "TEMPERATURE",
            "RMSE_48",
            "TSE_48"]

train_scaled = scaler.fit_transform(train[base_features])
val_scaled = scaler.transform(validation[base_features])
test_scaled = scaler.transform(test[base_features])


Feature scaling is not required for statistical models such as SARIMAX, as these models operate directly on the original scale of the data. However, scaling may be applied when training neural network-based models such as PatchTST to improve training stability and convergence.

## Section 14. Saving Preprocessed Datasets for Modelling

After completing the preprocessing and feature engineering steps, the final datasets are exported as CSV files for reuse in the modelling stage. This step ensures that the preprocessing pipeline does not need to be recomputed repeatedly when training different models. The processed dataset (df_model) and its corresponding train, validation, and test splits are saved separately. In addition, scaled versions of the datasets are also stored for use in deep learning models such as LSTM and PatchTST, which require normalized input features for stable training.

Saving these datasets improves workflow reproducibility and modularity. Subsequent modelling notebooks can directly load the processed data without rerunning the entire preprocessing procedure. This approach also reduces computational overhead and helps maintain consistency across different model experiments.

The following datasets are saved:

- df_model.csv: Final dataset after preprocessing and feature engineering

- train.csv: Training dataset

- validation.csv: Validation dataset

- test.csv: Test dataset

- train_scaled.csv: Scaled training dataset for neural network models

- val_scaled.csv: Scaled validation dataset

- test_scaled.csv: Scaled test dataset

These files are stored in the directory:

capstone_project_GroupA/data/NSW/

This structure allows different modelling notebooks (train.csv, test.csv and validation.csv for SARIMAX and train_scaled, test_scaled and val_scaled for LSTM, PatchTST) to load the same consistent datasets.

In [44]:
# Convert scaled numpy arrays into DataFrames so they can be saved as CSV files
# This preserves column names and datetime index for later modelling

base_features = ["TOTALDEMAND",
            "demand_1_day_ago",
            "demand_1_week_ago",
            "demand_1_year_ago",
            "TEMPERATURE",
            "RMSE_48",
            "TSE_48"]

train_scaled = pd.DataFrame(train_scaled, columns=base_features, index=train.index)
val_scaled = pd.DataFrame(val_scaled, columns=base_features, index=validation.index)
test_scaled = pd.DataFrame(test_scaled, columns=base_features, index=test.index)


# ---------------------------------------------------------
# Save all processed datasets for use in modelling notebooks
# ---------------------------------------------------------

import os

repo_path = "capstone_project_GroupA"
output_path = os.path.join(repo_path, "data", "NSW")

# Create folder if it does not exist
os.makedirs(output_path, exist_ok=True)

# Save main modelling dataset
df_model[base_features].to_csv(os.path.join(output_path, "df_model.csv"))

# Save train / validation / test splits
train.to_csv(os.path.join(output_path, "train.csv"))
validation.to_csv(os.path.join(output_path, "validation.csv"))
test.to_csv(os.path.join(output_path, "test.csv"))

# Save scaled datasets (used for LSTM and PatchTST models)
train_scaled.to_csv(os.path.join(output_path, "train_scaled.csv"))
val_scaled.to_csv(os.path.join(output_path, "val_scaled.csv"))
test_scaled.to_csv(os.path.join(output_path, "test_scaled.csv"))

# Display confirmation
print("Saved files to:", output_path)
print(os.listdir(output_path))

Saved files to: capstone_project_GroupA/data/NSW
['train.csv', 'validation.csv', 'forecastdemand_nsw.csv.zip', 'totaldemand_nsw.csv.zip', 'temperature_nsw.csv.zip', 'train_scaled.csv', 'forecastdemand_nsw.csv.zip.partaa', 'val_scaled.csv', 'test.csv', 'forecastdemand_nsw.csv.zip.partab', 'test_scaled.csv', 'df_model.csv']


The dataset has been:
- aligned to a consistent 30-minute timeline
- enriched with forecast horizons (P01–P48)
- cleaned using interpolation to handle missing values
- enhanced with lag-based features capturing daily, weekly, and yearly patterns
- split chronologically into training, validation, and test sets

The processed dataset is now ready for modelling and evaluation.